## Get nucleotide frequencies of Tuberculosis genome
Mycobacterium tuberculosis H37Rv

In [3]:
! wget "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_000962.3&rettype=fasta&retmode=text" -O data/NC_000962.3.fasta

--2025-06-27 12:28:01--  https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_000962.3&rettype=fasta&retmode=text
Resolving eutils.ncbi.nlm.nih.gov (eutils.ncbi.nlm.nih.gov)... 2607:f220:41e:4290::110, 130.14.29.110
Connecting to eutils.ncbi.nlm.nih.gov (eutils.ncbi.nlm.nih.gov)|2607:f220:41e:4290::110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘data/NC_000962.3.fasta’

data/NC_000962.3.fa     [    <=>             ]   4,27M  5,27MB/s    in 0,8s    

2025-06-27 12:28:03 (5,27 MB/s) - ‘data/NC_000962.3.fasta’ saved [4474618]



In [14]:
from collections import Counter
import numpy as np
import pandas as pd
from Bio import SeqIO
import sys

In [12]:
def calculate_nucleotide_frequencies(fasta_path):
    """
    Calculates nucleotide fequencies of a genome.
    Would also work for a fasta file with more than one sequence, even though I'm using it for a whole genome fasta file.
    """
    total_counts = Counter()

    for record in SeqIO.parse(fasta_path, "fasta"):
        seq = str(record.seq).upper()
        total_counts.update(seq)

    total_nucleotides = sum(total_counts[nuc] for nuc in "ACGT")
    frequencies = {nuc: total_counts[nuc] / total_nucleotides for nuc in "ACGT"}

    print("Nucleotide Frequencies:")
    for nuc in "ACGT":
        print(f"{nuc}: {frequencies[nuc]:.4f}")
    return frequencies

freq = calculate_nucleotide_frequencies("data/NC_000962.3.fasta")

Nucleotide Frequencies:
A: 0.1719
C: 0.3287
G: 0.3275
T: 0.1719


In [17]:
def create_hky_q_matrix(frequencies, kappa):
    """
    Creates the scaled HKY Q matrix such that the expected substitution rate is 1.
    Rows and columns are ordered as A, C, G, T.
    """
    nucs = ['A', 'C', 'G', 'T']
    pi = [frequencies[nuc] for nuc in nucs]

    # Initialize Q matrix
    Q = np.zeros((4, 4))

    # Define transitions and transversions
    transitions = {('A', 'G'), ('G', 'A'), ('C', 'T'), ('T', 'C')}

    for i, ni in enumerate(nucs):
        for j, nj in enumerate(nucs):
            if i == j:
                continue
            if (ni, nj) in transitions:
                Q[i, j] = kappa * frequencies[nj]
            else:
                Q[i, j] = frequencies[nj]

    # Set diagonal entries so rows sum to zero
    for i in range(4):
        Q[i, i] = -np.sum(Q[i, :])

    # Scale so that the expected substitution rate is 1
    total_rate = -sum(pi[i] * Q[i, i] for i in range(4))
    Q /= total_rate

    return Q, pd.DataFrame(Q, index=nucs, columns=nucs)

In [23]:
Q, Q_df = create_hky_q_matrix(freq, kappa = 2.07)
Q

array([[-1.21861714,  0.3398901 ,  0.70096006,  0.17776698],
       [ 0.17781012, -0.88441582,  0.33862805,  0.36797766],
       [ 0.36806694,  0.3398901 , -0.88572402,  0.17776698],
       [ 0.17781012,  0.7035725 ,  0.33862805, -1.22001066]])

In [24]:
# scale Q by substitution rate
mu = 4.6e-8
Q = mu * Q
Q

array([[-5.60563883e-08,  1.56349444e-08,  3.22441626e-08,
         8.17728129e-09],
       [ 8.17926531e-09, -4.06831277e-08,  1.55768902e-08,
         1.69269723e-08],
       [ 1.69310792e-08,  1.56349444e-08, -4.07433049e-08,
         8.17728129e-09],
       [ 8.17926531e-09,  3.23643349e-08,  1.55768902e-08,
        -5.61204904e-08]])

Seq-Gen interprets branch lengths as units of expected substitutions per site. If a branch length is 0.1, Seq-Gen treats it as 0.1 expected substitutions per site.

Suppose: A branch length in your tree is 1.2 years. The substitution rate is μ=4.6×10−8 substitutions/site/year. To tell Seq-Gen: “this branch evolves for 1.2 years at this rate,” you compute: Substitutions/site on this branch=μ⋅time=4.6×10−8⋅1.2=5.52×10−8
→ This is the value that Seq-Gen needs as the branch length